In [1]:
import zarr
import numpy as np
from zarr.storage import LMDBStore


In [16]:
ls -lh /home/eirikmagnussen/Projects/microplastics/microscopy_prediction/outputs/

total 4,0K
drwxr-xr-x 2 root root 4,0K mars  24 16:37 predictions.lmdb/


In [21]:
zarr_preds_path = '/home/eirikmagnussen/Projects/microplastics/microscopy_prediction/outputs/predictions.lmdb'
store = LMDBStore(zarr_preds_path, 
                  map_size=int(1e12), readonly=True)
store = zarr.open_group(store, mode="r")

/tmp/ipykernel_5585/327243621.py:2: FutureWarning: The LMDBStore is deprecated and will be removed in a Zarr-Python version 3, see https://github.com/zarr-developers/zarr-python/issues/1274 for more information.
  store = LMDBStore(zarr_preds_path,


ReadonlyError: /home/eirikmagnussen/Projects/microplastics/microscopy_prediction/outputs/predictions.lmdb: Permission denied

In [18]:
import zarr
import numpy as np
from zarr.storage import LMDBStore


def open_pred_store_readonly(path: str) -> zarr.Group:
    store = LMDBStore(path, map_size=int(1e12), readonly=True)
    return zarr.open_group(store, mode="r")


def list_trials(store: zarr.Group) -> list[str]:
    """List all trial ids in the store, e.g. ['N8_seed00', 'N16_seed01']"""
    return sorted(store.keys())


def list_images(store: zarr.Group, trial_id: str) -> list[str]:
    """List all image names for a given trial."""
    return sorted(store[trial_id].keys())


def load_image_outputs(
    store: zarr.Group,
    trial_id: str,
    image_name: str,
    load_top_k: bool = False,
) -> dict:
    """
    Load saved inference outputs for a single image from a single trial.

    Returns
    -------
    dict with keys:
        argmax_map   : np.ndarray (H, W)       uint8
        bg_prob      : np.ndarray (H, W)       float16
        top_k_classes: np.ndarray (H, W, k)   uint8    — only if load_top_k=True
        top_k_probs  : np.ndarray (H, W, k)   float16  — only if load_top_k=True
        attrs        : dict  — all metadata (N, seed, true_idx, H, W, ...)
    """
    grp = store[f"{trial_id}/{image_name}"]

    result = {
        "argmax_map": grp["argmax_map"][:],
        "bg_prob":    grp["bg_prob"][:],
        "attrs":      dict(grp.attrs),
    }

    if load_top_k:
        result["top_k_classes"] = grp["top_k_classes"][:]
        result["top_k_probs"]   = grp["top_k_probs"][:]

    return result


def load_all_results(
    store:          zarr.Group,
    bg_threshold:   float = 0.5,
    trials:         list[str] | None = None,   # None = all trials
    load_top_k:     bool = False,
) -> list[dict]:
    """
    Load and threshold-filter all images across all trials.
    Returns a flat list of result dicts ready for metric computation.

    Each dict contains:
        trial_id, image_name, N, seed, true_idx,
        preds_masked, bg_prob_masked, mask, n_kept, n_total
    """
    trials     = trials or list_trials(store)
    results    = []

    for trial_id in trials:
        for image_name in list_images(store, trial_id):
            data     = load_image_outputs(store, trial_id, image_name, load_top_k=load_top_k)
            attrs    = data["attrs"]

            flat_argmax = data["argmax_map"].flatten()
            flat_bg     = data["bg_prob"].flatten().astype(np.float32)
            mask        = flat_bg <= bg_threshold

            entry = {
                "trial_id":       trial_id,
                "image_name":     image_name,
                "N":              attrs.get("N"),
                "seed":           attrs.get("seed"),
                "true_idx":       attrs.get("true_idx"),
                "preds_masked":   flat_argmax[mask],
                "bg_prob_masked": flat_bg[mask],
                "mask":           mask,
                "n_kept":         int(mask.sum()),
                "n_total":        int(mask.shape[0]),
                "bg_threshold":   bg_threshold,
            }

            if load_top_k:
                H, W, k = data["top_k_classes"].shape
                entry["top_k_classes"] = data["top_k_classes"].reshape(-1, k)[mask]
                entry["top_k_probs"]   = data["top_k_probs"].reshape(-1, k)[mask]

            results.append(entry)

    return results

In [19]:
# list trials and images
trials = sorted(store.keys())
print("trials:", trials)
# → ['N8_seed00', 'N8_seed01', ...]

images = sorted(store["N8_seed00"].keys())
print("images:", images)
# → ['ABS', 'PA12', 'PEHD', 'PET', 'PHB', 'PS', 'PVAC', 'PVC']

# load a single image
grp      = store["N8_seed00/ABS"]
argmax   = grp["argmax_map"][:]        # (640, 512)
bg_prob  = grp["bg_prob"][:]           # (640, 512)
top_k_c  = grp["top_k_classes"][:]    # (640, 512, 3)
top_k_p  = grp["top_k_probs"][:]      # (640, 512, 3)
attrs    = dict(grp.attrs)
print("attrs:", attrs)

# threshold sweep across all trials and images
rows = []
for trial_id in sorted(store.keys()):
    for image_name in sorted(store[trial_id].keys()):
        grp      = store[f"{trial_id}/{image_name}"]
        attrs    = dict(grp.attrs)
        argmax   = grp["argmax_map"][:].flatten()
        bg_prob  = grp["bg_prob"][:].flatten().astype(np.float32)
        true_idx = attrs.get("true_idx")
        N        = attrs.get("N")
        seed     = attrs.get("seed")

        for threshold in [0.3, 0.5, 0.7, 0.9]:
            mask  = bg_prob <= threshold
            preds = argmax[mask]
            acc   = float(np.mean(preds == true_idx)) if mask.sum() > 0 else np.nan

            rows.append({
                "trial_id":  trial_id,
                "N":         N,
                "seed":      seed,
                "image":     image_name,
                "threshold": threshold,
                "accuracy":  acc,
                "n_kept":    int(mask.sum()),
                "pct_kept":  float(mask.sum() / mask.shape[0]),
            })

df = pd.DataFrame(rows)
print(df.groupby(["N", "threshold"])["accuracy"].agg(["mean", "std"]).round(4))
print(df.groupby(["image", "threshold"])["accuracy"].mean().unstack().round(4))

NameError: name 'store' is not defined